# Launching a pulse on the ZCU216 — `RemoteDriver` quickstart

The **hardware** counterpart of [`amplitude_sweep.ipynb`](amplitude_sweep.ipynb): connect to the
board server running on the ZCU216's ARM, load a gateware bundle, compile a small on-core
`@kernel` that fires a train of gate pulses, and run it — the pulses come out of a real DAC.

The host code is byte-identical to the co-sim notebooks; only the driver constructor changes
(`RemoteDriver` instead of `riscq.sim.server.start`). Prerequisite: the board server is up —
see [docs/software/board-server.md](../docs/software/board-server.md) for the one-time setup
(offline install, building the `top.xsa`, starting `riscq-board-server`).

This notebook is a hardware walkthrough, so unlike the co-sim examples it is **not executed in
CI** (it needs a ZCU216 on the LAN).

In [ ]:
import numpy as np

from riscq import run as rq
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, ParamTable, compile_kernel, kernel
from riscq.map import LEAD, SocMap, SocParams
from riscq.pulses import Pulse, envelopes, units

BOARD = "192.168.1.122"                   # the ZCU216's LAN address (or the full PYRO: uri)

drv = RemoteDriver(BOARD, 9091)
print("server:", drv.board.info())

## The bundle

Gateware reaches the offline board as a **bundle** — `top.xsa` + the `params.json` that
parameterized the build (+ an optional `board.json`) — uploaded over this same connection and
kept in the board-side store. This example uses the 2-core `xm650-loopback` build
([`software/configs/xm650-loopback.json`](../software/configs/xm650-loopback.json)): core 0's gate + readout drive sum
onto **DAC 0** (`vout00`) and its demod listens on **ADC 14** (`vin32`), so a cable between the
two closes the loop.

The upload is needed **once per build** (chunked, sha256-verified); `load` is needed once per
server start (it downloads the bitstream and runs the RF bring-up: ref clocks → overlay → MTS →
Nyquist zones). Skip both if the server already reports the bundle loaded above.

In [ ]:
print("bundles on the board:", drv.board.bundles())

# first time only — push the build up and load it (~100 MB, a minute on GbE):
# upload_bundle(drv, "xm650-loopback",
#               xsa="../build/xm650-loopback/top.xsa",                    # write_hw_platform export
#               params_json="../software/configs/xm650-loopback.json")   # the SAME JSON the build used
# info = drv.board.load("xm650-loopback")                                # full RF bring-up; returns info()
# print(info)
# assert info["mts_result"] == 0, "multi-tile sync missed its target latencies"

m = SocMap(SocParams.from_json(drv.board.get_params()))   # always matches the loaded bitstream
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{units.sample_rate(m.params) / 1e9:.0f} GS/s DACs")

## The kernel

Identical DSL to the co-sim notebooks (see `amplitude_sweep.ipynb` for the full tour): the pulse
lives in a `ParamTable` on channel 0 (this core's gate drive), `init_pulse_params` loads it into
the hardware slots, and each loop iteration `play`s it `LEAD` batches into the future, recording
the fire time in `ts` for the host. `n`, `amp`, `gap` stay runtime parameters — re-runnable
without recompiling.

In [ ]:
N = 8                                     # pulses in the train
F_DRIVE = 80e6                            # carrier frequency (Hz)


@kernel
def pulse_train(gate: ParamTable, ts: Array, n: int, amp: int, gap: int):
    """Fire `n` identical gate pulses, `gap` idle batches apart, recording each fire time."""
    init_pulse_params(gate.pulses)                     # noqa: F821  load the pulse table into HW
    set_freq(gate, gate.freq)                          # noqa: F821  program the carrier
    for i in range(n):
        set_amp(gate, gate["x90"], amp)                # noqa: F821
        t = now() + LEAD                               # noqa: F821  schedule LEAD batches ahead
        ts[i] = t
        play(gate, gate["x90"], t)                     # noqa: F821
        wait_until(t + gate["x90"].dur + gap)          # noqa: F821  hold past the pulse + gap


pulse = Pulse(envelopes.cos_edge_square(800, 0.2), freq_hz=F_DRIVE, amp=1.0)
gate = ParamTable(0, F_DRIVE, {"x90": pulse})
prog = compile_kernel(pulse_train, m, tables=dict(gate=gate), ts=Array(N))

dur = len(pulse.packed_lines(m, "gate"))               # pulse length in batches
print(f"compiled: {N} pulses of {dur} batches ({units.ns(dur, m.params):.0f} ns) "
      f"at {F_DRIVE / 1e6:.0f} MHz")

## Run it

`rq.run` = `setup` (load the image + envelopes + pulse table, park the unused core) + one
`rerun` (write params → release reset → poll DONE → read results). `RemoteDriver` routes both
**server-side, one RPC each** — the whole batch crosses the LAN in 2 round trips.

On hardware `timeout` counts ~1 ms polls (not sim cycles): `5_000` ≈ a 5 s deadline. The kernel
finishes in microseconds; the pulses appear on DAC 0 / `vout00` — with the loopback cable (or a
scope) you can see the 80 MHz bursts directly.

In [ ]:
out = rq.run(drv, m, {0: prog},
             params={0: dict(n=N, amp=units.amp_to_code(0.8), gap=8)},
             results=["ts"], timeout=5_000)

ts = out[0]["ts"][:N]
spacing_ns = [float(units.ns(int(d), m.params)) for d in np.diff(ts)]
print("fire times (batches):", [int(t) for t in ts])
print("pulse spacing (ns):  ", [f"{s:.0f}" for s in spacing_ns])

## Re-run without reloading

The program stays resident: further `rq.rerun` calls skip the image/envelope load and just write
new parameter values — retune the amplitude, spacing, or count at will, one RPC per run.

In [ ]:
for amp in (0.2, 0.5, 0.9):
    out = rq.rerun(drv, m, {0: prog},
                   params={0: dict(n=N, amp=units.amp_to_code(amp), gap=8)},
                   results=["ts"], timeout=5_000)
    print(f"amp {amp:.1f}: fired {N} pulses, first at batch {int(out[0]['ts'][0])}")

drv.close()
print("done")